In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import torch

In [ ]:
cols = ["letter", "x-box", "y-box", "width", "high", "onpix", "x-bar", "y-bar", "x2bar", "y2bar", "xybar", "x2ybr", "xy2br", "x-ege", "xegvy", "y-ege", "yegv"]

ds = pd.read_csv("letter+recognition/letter-recognition.data", header=None)
ds.columns = cols
ds.head()

In [ ]:
ds.describe()

In [ ]:
sns.countplot(data=ds, x="letter")

In [ ]:
from sklearn.model_selection import train_test_split

features = ds.drop(["letter"], axis=1)
labels = ds["letter"]

# normalize the features
features = (features - features.mean()) / features.std()
features = features.values.astype(np.float32)
print("features: ", features.shape, features.dtype)

# one-hot encode the labels
labels = pd.get_dummies(labels, dtype=np.float32)
labels = labels.values
print("labels: ", labels.shape, labels.dtype)

# split the dataset into training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.33)
print("X_train: ", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train: ", y_train.shape)
print("y_test: ", y_test.shape)

# convert to PyTorch Tensor(s)
X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

n_features = X_train.shape[1]
n_classes = y_train.shape[1]

In [ ]:
# define data loaders
# TensorDataset docs: https://pytorch.org/docs/stable/data.html
# DataLoader docs: https://pytorch.org/tutorials/beginner/basics/data_tutorial.html#preparing-your-data-for-training-with-dataloaders
train_ds = torch.utils.data.TensorDataset(X_train, y_train)
test_ds = torch.utils.data.TensorDataset(X_test, y_test)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=32)

# define a sequential model (nn.Module)
# use PyTorch's graph compilation strategy to speedup computations
# docs: https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html#
model = torch.nn.Sequential(
    torch.nn.Linear(n_features, 32),
    torch.nn.LeakyReLU(),
    torch.nn.Linear(32, 64),
    torch.nn.LeakyReLU(),
    torch.nn.Linear(64, n_classes),
)
# (optional) use PyTorch's graph compilation strategy to speedup computations
# docs: https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html#
@torch.compile
def execute_model():
    inp = torch.randn(1, n_features)
    for layer in model:
        inp = layer(inp)
execute_model()

# CrossEntropyLoss expects unnormalized logits as input
# docs: https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
ce_loss = torch.nn.CrossEntropyLoss()

# the adam optimizer
# docs: https://pytorch.org/docs/stable/generated/torch.optim.Adam.html
optimizier = torch.optim.Adam(model.parameters())

# define training and testing steps
def train_step(batch):
    batch_X, batch_y = batch
    optimizier.zero_grad()
    logits = model(batch_X)
    batch_loss = ce_loss(logits, batch_y)
    batch_loss.backward()
    optimizier.step()
    return batch_loss.item()

def test_step(batch):
    batch_X, batch_y = batch
    logits = model(batch_X)
    batch_loss = ce_loss(logits, batch_y)
    return batch_loss.item()
    
def train_epoch():
    mean_train_loss = 0.0
    for i, batch in enumerate(train_loader):
        batch_train_loss = train_step(batch)
        mean_train_loss += batch_train_loss
    mean_train_loss = mean_train_loss / len(train_loader)
        
    mean_test_loss = 0.0
    for i, batch in enumerate(test_loader):
        batch_test_loss = test_step(batch)
        mean_test_loss += batch_test_loss
    mean_test_loss = mean_test_loss / len(test_loader)
    return mean_train_loss, mean_test_loss
    
def predict_classes(ds):
    predictions = []
    model.eval()
    for (sample, _) in ds:
        logits = model(sample)
        pred_class = torch.argmax(logits).item()
        predictions.append(pred_class)
    return predictions

In [ ]:
from sklearn.metrics import accuracy_score

# transform from one-hot labels to integer labels
# ex. [0 0 1] => [2] where n_classes = 3
y_train_classes = np.argmax(y_train, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

n_epochs = 50
for _ in range(n_epochs):
    train_loss, test_loss = train_epoch()
    train_pred_classes = predict_classes(train_ds)
    test_pred_classes = predict_classes(test_ds)
    train_acc = accuracy_score(y_train_classes, train_pred_classes)
    test_acc = accuracy_score(y_test_classes, test_pred_classes)
    print(f"train_loss: {train_loss}\ttest_loss: {test_loss}\ttrain_acc: {train_acc}\ttest_acc: {test_acc}")

In [ ]:
test_pred_classes = predict_classes(test_ds)
test_acc = accuracy_score(y_test_classes, test_pred_classes)
print(f"ttest_loss: {test_loss}\ttest_acc: {test_acc}")